# 04 — Analyse results

Loads the CSVs written by notebook 03 and renders the cross-scenario tables and plots used in Chapters 5 and 6 of the thesis.

Headline metrics now include:
- started vs. completed charging events,
- % of agents who charged at least once,
- % of agents who waited at least once,
- median / 95th-percentile waiting time,
- mean waiting time *among waiters only*,
- total queue-minutes and max queue length.

In [ ]:
import sys
from pathlib import Path
PROJECT = Path.cwd().resolve().parent
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))

import pandas as pd
import matplotlib.pyplot as plt
from src import config

In [ ]:
summary = pd.read_csv(config.TABLES_DIR / 'scenario_summary.csv')
agents = pd.read_csv(config.TABLES_DIR / 'agent_results.csv')
stations = pd.read_csv(config.TABLES_DIR / 'station_results.csv')
summary.T  # transpose so all metrics fit vertically

### Per-scenario aggregates (agent-level)

In [ ]:
agents.groupby('scenario').agg(
    completed_trips=('completed_trips', 'sum'),
    started_charging=('started_charging_events', 'sum'),
    completed_charging=('completed_charging_events', 'sum'),
    times_queued=('times_queued', 'sum'),
    mean_wait=('waiting_time_min', 'mean'),
    median_wait=('waiting_time_min', 'median'),
    p95_wait=('waiting_time_min', lambda s: s.quantile(0.95)),
    max_wait=('waiting_time_min', 'max'),
    mean_detour_m=('detour_distance_m', 'mean'),
    median_detour_m=('detour_distance_m', 'median'),
    stranded=('stranded', 'sum'),
)

### Waiters only — the meaningful waiting-time view

In [ ]:
waiters = agents[agents['waiting_time_min'] > 0]
waiters.groupby('scenario')['waiting_time_min'].describe()[['count', 'mean', '50%', '95%', 'max']]

### Per-scenario aggregates (station-level)

In [ ]:
stations.groupby('scenario').agg(
    n_stations=('station_id', 'count'),
    total_ports=('n_ports', 'sum'),
    mean_util=('utilisation', 'mean'),
    max_util=('utilisation', 'max'),
    total_queue_min=('total_queue_minutes', 'sum'),
    max_queue=('max_queue', 'max'),
    total_started=('total_started', 'sum'),
    total_completed=('total_completed', 'sum'),
)

### Pre-rendered figures (from notebook 03)

In [ ]:
from IPython.display import Image, display
for name in [
    'queue_over_time.png',
    'scenario_comparison_waiting_time.png',
    'waiting_time_waiters_only.png',
    'detour_distribution.png',
    'station_utilisation.png',
    'utilisation_histogram.png',
    'charging_events_by_scenario.png',
]:
    p = config.FIGURES_DIR / name
    if p.exists():
        display(Image(filename=str(p)))
    else:
        print(f'missing: {p}')